# Bubble evolution within one train

Follows **every frame of one train** (248 frames, 886 ns apart ≈ 220 µs) and produces the same analyses as
`bubble_analysis.ipynb`, as a function of time. It also makes a GIF of the area-coloured bubbles.

**Tracking (section 2)** follows each bubble from frame to frame, so every bubble keeps one identity (`track_id`)
through the train. This works for deformed, non-elliptical bubbles too, because it compares outlines by their
overlap. Each bubble is moved by its recent motion before comparing. A bubble the model misses for up to
`max_gap` (6) frames keeps its identity. Those frames are filled with the model's own low-score detection at that
position if there is one ("rescued"), otherwise with an outline morphed between the frames before and after
("interpolated"). Where a bubble ends or starts is classified: merged into an overlapping bubble (volume conserved),
dissolved, left the view, appeared, and so on (see `bubble_track.py`). By default all analyses use the tracked
bubbles, interpolated ones included.

Set `RUN` and `TRAIN` in CONFIG. Everything for that train goes into the run's results folder, `results/<RUN>/tid<TRAIN>/`
(e.g. `results/r563/tid3/`):

| file | content |
|---|---|
| `shapes/` | frame + bubble outlines per frame (model drafts, editable in napari) |
| `candidates/` | the model's low-score detections per frame (used only to bridge missed frames) |
| `tracked/` | tracking result: `shapes/` (all outlines with track id), `bubbles.csv`, `frames.csv`, `tracks.csv` (one row per bubble), `events.csv` (mergers, dissolutions, ...) |
| `bubbles.csv`, `frames.csv` | untracked measurements (only with `USE_TRACKING = False`) |
| `frame_metrics.csv` | per-frame metrics (section 4) |
| `figures/` | all plots, `overlays_area.gif` and `tracks.gif` |

**Section 1 predicts all frames with your model.** Frames that already have a draft are not predicted again, except
once to save their low-score detections if they have none yet. Reviewing 248 frames by hand is not practical, so unreviewed frames are included by default
(`INCLUDE_UNREVIEWED`). Any frames you do correct in napari (`python annotate_bubbles.py results/r563/tid3/shapes`)
are used automatically when you re-run sections 1 and 2.

Within a train, frame-to-frame changes are partly measurement noise (the model is not identical on two almost identical
frames). The line plots therefore also show a running mean (`SMOOTH` frames).

In [ ]:
import os, time, dataclasses
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2
import bubble_rcnn as br
import bubble_io as bio
import bubble_stats as bst
import bubble_track as bt

bst.style()

## CONFIG

In [ ]:
RUN = "r563"                 # run folder in results/ (same name as in bubble_inference.ipynb)
TRAIN = 3
RUN_DIR = os.path.join("results", RUN, f"tid{TRAIN}")
FRAME_GLOB = "raw/jpg_r563_svd_normalised_tr050/*.jpg"
MODEL_PATH = "models/bubble_rcnn_best.pt"

# --- section 1: prediction ---
RUN_PREDICTION = True        # predict frames that have no draft yet (needs the GPU kernel)
OVERWRITE_UNREVIEWED = False # True: predict again frames that are not reviewed (e.g. after retraining or changing DRAFT_KIND)
CANDIDATE_SCORE = 0.15       # also save low-score detections (score CANDIDATE_SCORE ... score_thresh) so tracking can
                             # bridge frames where a bubble was missed; None = off (gaps are then only interpolated)
INCLUDE_UNREVIEWED = True    # use frames you have not ticked "Reviewed"
DRAFT_KIND = "auto"           # circle, else ellipse, polygon only if neither fits ("polygon" / "ellipse": always that)
INFER = dict(                # same meaning as in bubble_inference.ipynb
    score_thresh=0.5, mask_thresh=0.5, mask_nms_iou=0.7, um_per_px=3.2,
    edge_min_arc_deg=100.0, edge_ellipse_gain=1.6, edge_max_axis_ratio=2.5, edge_reliable_arc_deg=140.0,
    drop_unreliable_fits=False, drop_edge_bubbles=False, inner_margin=0.0,
)

# --- section 2: tracking ---
USE_TRACKING = True          # analyse the tracked bubbles (False: every frame on its own, as before)
RUN_TRACKING = True          # (re)run the tracking (~5 min for 248 frames, mostly measuring the outlines)
INCLUDE_INTERPOLATED = True  # count bubbles bridged with a morphed outline in the statistics
MIN_TRACK_FRAMES = 1         # leave out bubbles detected in fewer frames (1 = keep all; 3 removes most one-off
                             # false detections, which tracking labels 'transient')
TRACK = dict(                # see bubble_track.TrackConfig for all options
    max_gap=6,               # frames a bubble may be missed and keep its identity
    min_iou=0.3,             # outline overlap needed to link a bubble to the next frame
    rescue_min_iou=0.5,      # a low-score detection must overlap the expected outline this much
    merge_min_cover=0.4,     # an ending bubble this much inside another one -> merged / occluded
    min_track_frames=3,      # shorter tracks are 'transient' (no events)
)
RUN_EXPORT = not USE_TRACKING  # untracked measurements -> bubbles.csv / frames.csv
# --- analysis ---
FRAME_INTERVAL_US = 0.886    # HPV-X2 framing time (886 ns)
TIME_LABEL = "time [µs]"
UM_PER_PX = 3.2
SMOOTH = 5                   # running mean over this many frames in the line plots (1 = off)
N_SNAPSHOTS = 8              # frames shown in histograms / overlay grid / density maps (evenly spaced)
COMPUTE_COVERAGE = True      # union of outlines per frame (~1 min for 248 frames)
HIST_VAR, HIST_LABEL = "r_eq_um", "equivalent radius [µm]"
SIZE_CLASSES_UM = [0, 10, 20, 40, np.inf]
OVERLAY_CMAP = "RdYlGn_r"    # green (small) -> red (large); "viridis" is colour-blind safe
OVERLAY_ALPHA = 0.45
DENSITY_CELL_UM = 80

# --- individual bubbles (tracking) ---
MIN_FRAMES_PATH = 20         # bubbles followed at least this many frames are drawn in the path / growth plots
EVENT_BIN_FRAMES = 20        # time bins for the event rates

# --- GIF ---
MAKE_GIF = True
MAKE_TRACK_GIF = True        # second GIF: every bubble keeps its own colour (shows the tracking)
GIF_EVERY = 1                # use every n-th frame (2 = half the frames, half the file size)
GIF_FPS = 12
GIF_DPI = 80                 # resolution of the GIF frames

SAVE_FIGS = True
FIG_DIR = os.path.join(RUN_DIR, "figures")

def save(fig, name):
    if SAVE_FIGS:
        os.makedirs(FIG_DIR, exist_ok=True)
        fig.savefig(os.path.join(FIG_DIR, name + ".png"), dpi=200, bbox_inches="tight")

## 1. Predict all frames of the train

In [ ]:
mcfg = dataclasses.replace(br.RCNNConfig.from_json(MODEL_PATH.replace(".pt", "_config.json")), **INFER)

if RUN_PREDICTION:
    import torch
    print("CUDA:", torch.cuda.is_available())
    index = bio.index_frames(FRAME_GLOB)
    paths = bio.select_frames([(TRAIN, "all")], index)
    model, _ = br.load_model(MODEL_PATH)
    bio.write_run_info(RUN_DIR, MODEL_PATH, mcfg, extra=dict(train=TRAIN))
    t0 = time.time()
    summary = bio.draft_frames(model, mcfg, paths, RUN_DIR, model_path=MODEL_PATH, kind=DRAFT_KIND, show_every=25,
                               candidate_thresh=CANDIDATE_SCORE, overwrite_unreviewed=OVERWRITE_UNREVIEWED)
    print(f"prediction done in {time.time() - t0:.0f}s:", summary["status"].value_counts().to_dict())

if RUN_EXPORT:
    t0 = time.time()
    bio.export_results(RUN_DIR, mcfg, include_unreviewed=INCLUDE_UNREVIEWED)
    print(f"export done in {time.time() - t0:.0f}s")

## 2. Track the bubbles through the train

The printout shows how many detections were linked, and how many missed frames were bridged. `tracks` has one row
per bubble and `events` one row per start or end. Transient tracks are bubbles detected in fewer than
`min_track_frames` frames, mostly one-off false detections.

In [ ]:
if USE_TRACKING and RUN_TRACKING:
    bt.track_run(RUN_DIR, mcfg, bt.TrackConfig(**TRACK), include_unreviewed=INCLUDE_UNREVIEWED,
                 frame_interval_us=FRAME_INTERVAL_US)
if USE_TRACKING:
    _, tframes, tracks, events = bt.load_tracking(RUN_DIR)
    tframes["t"] = tframes["frame_idx"] * FRAME_INTERVAL_US
    real = tracks[tracks["end_event"] != "transient"]
    print(f"{len(tracks)} tracks: {len(real)} bubbles + {len(tracks) - len(real)} transient | "
          f"{(real['n_frames'] >= 0.9 * len(tframes)).sum()} followed through ≥ 90% of the train | "
          f"bridged: {tframes['n_rescued'].sum()} rescued, {tframes['n_interpolated'].sum()} interpolated")
    display(pd.concat([real["start_event"].value_counts().rename("start"),
                       real["end_event"].value_counts().rename("end")], axis=1).fillna(0).astype(int))

## 3. Load

In [ ]:
DATA_DIR = bt.tracked_dir(RUN_DIR) if USE_TRACKING else RUN_DIR
bubbles, frames = bio.load_results(DATA_DIR, reviewed_only=not INCLUDE_UNREVIEWED, in_analysis_only=True)
if USE_TRACKING:
    if not INCLUDE_INTERPOLATED:
        bubbles = bubbles[bubbles["source"] != "interpolated"]
    keep = tracks.loc[tracks["n_detected"] + tracks["n_rescued"] >= MIN_TRACK_FRAMES, "track_id"]
    bubbles = bubbles[bubbles["track_id"].isin(keep)].reset_index(drop=True)
bubbles["t"] = bubbles["frame_idx"] * FRAME_INTERVAL_US
frames["t"] = frames["frame_idx"] * FRAME_INTERVAL_US
frames = frames.sort_values("t").reset_index(drop=True)
NATIVE_HW = (int(frames["image_height_px"].iloc[0]), int(frames["image_width_px"].iloc[0]))
print(f"train {TRAIN}: {len(frames)} frames ({frames['reviewed'].sum()} reviewed), {len(bubbles)} bubbles, "
      f"t = {frames['t'].min():.1f} … {frames['t'].max():.1f} µs")
frame_label = lambda r: f"{r['t']:.1f} µs (frame {int(r['frame_idx'])})"

## 4. Metrics per frame

In [ ]:
t0 = time.time()
m = bst.frame_metrics(bubbles, frames, run_dir=DATA_DIR if COMPUTE_COVERAGE else None, um_per_px=UM_PER_PX)
m.to_csv(os.path.join(RUN_DIR, "frame_metrics.csv"), index=False)
snap = bst.select_snapshots(m, N_SNAPSHOTS)
print(f"metrics for {len(m)} frames in {time.time() - t0:.0f}s; snapshots at frames {snap['frame_idx'].tolist()}")
cols = ["frame_idx", "t", "n_bubbles", "density_per_mm2", "r_mean_um", "r_median_um", "r32_um", "r_max_um",
        "polydispersity", "area_total_um2", "volume_um3", "frac_noncircular"] + (["coverage_frac"] if COMPUTE_COVERAGE else [])
display(m[cols].describe().loc[["mean", "std", "min", "max"]].round(3))

## 5. Bubble count

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
bst.plot_count(m, TIME_LABEL, ax=ax, smooth=SMOOTH)
save(fig, "count")

## 6. Average bubble size

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.2))
bst.plot_mean_size(m, TIME_LABEL, ax=ax, smooth=SMOOTH)
save(fig, "mean_size")

## 7. Size distributions

The heatmap shows every frame's size distribution as one column. A band drifting upwards means growth, and
fading low bands mean small bubbles disappearing. Below it are histograms, cumulative and scaled distributions for
`N_SNAPSHOTS` evenly spaced frames.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.2))
bst.plot_size_heatmap(bubbles, m, TIME_LABEL, var=HIST_VAR, label=HIST_LABEL, ax=ax)
save(fig, "size_heatmap")

In [ ]:
fig = bst.plot_histograms(bubbles, snap, var=HIST_VAR, xlabel=HIST_LABEL, frame_label=frame_label)
save(fig, "histograms")
fig, ax = plt.subplots(1, 2, figsize=(14, 4.2))
bst.plot_cdfs(bubbles, snap, var=HIST_VAR, xlabel=HIST_LABEL, ax=ax[0], frame_label=frame_label)
bst.plot_scaled_distribution(bubbles, snap, ax=ax[1], frame_label=frame_label)
fig.tight_layout()
save(fig, "distributions")

## 8. Size classes and coalescence check

In [ ]:
counts, labels = bst.size_class_counts(bubbles, SIZE_CLASSES_UM)
fig, ax = plt.subplots(1, 2, figsize=(15, 4.2))
bst.plot_size_classes(counts, labels, TIME_LABEL, ax=ax[0], smooth=SMOOTH)
bst.plot_relative(m, TIME_LABEL, ax=ax[1], smooth=SMOOTH)
fig.tight_layout()
save(fig, "size_classes_relative")

## 9. Overlays coloured by bubble area + GIF

One shared logarithmic colour scale for all frames (green = small, red = large). The GIF contains every
`GIF_EVERY`-th frame. With 248 frames it takes a few minutes and is roughly 10–30 MB; use `GIF_EVERY = 2` or a lower
`GIF_DPI` to make it smaller.

In [ ]:
norm = bst.area_norm(bubbles, "area_um2")
ncols = 2
nrows = int(np.ceil(len(snap) / ncols))
fig, axs = plt.subplots(nrows, ncols, figsize=(8 * ncols, 5.2 * nrows), squeeze=False)
for ax, (_, r) in zip(axs.ravel(), snap.iterrows()):
    img, shapes = bio.load_frame(DATA_DIR, r["frame"])
    b = bubbles[bubbles["frame"] == r["frame"]]
    bst.plot_area_overlay(img, [shapes[i] for i in b["bubble_id"]], b["area_um2"].to_numpy(), norm,
                          cmap=OVERLAY_CMAP, ax=ax, alpha=OVERLAY_ALPHA, title=f"{frame_label(r)}: {len(b)} bubbles")
for ax in axs.ravel()[len(snap):]:
    ax.set_visible(False)
fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=OVERLAY_CMAP), ax=axs.ravel().tolist(), shrink=0.6,
             label="bubble area [µm²]")
save(fig, "overlays_area")

In [ ]:
if MAKE_GIF:
    t0 = time.time()
    gif_path = os.path.join(FIG_DIR, "overlays_area.gif")
    path, n = bst.make_overlay_gif(DATA_DIR, bubbles, m, norm, gif_path, cmap=OVERLAY_CMAP, alpha=OVERLAY_ALPHA,
                                   every=GIF_EVERY, fps=GIF_FPS, dpi=GIF_DPI,
                                   time_fmt=lambda r: f"train {TRAIN}  ·  {frame_label(r)}")
    print(f"GIF with {n} frames: {path}  ({os.path.getsize(path) / 1e6:.1f} MB, {time.time() - t0:.0f}s)")

## 10. Spatial distribution and drift

In [ ]:
fig = bst.plot_density_maps(bubbles, snap, NATIVE_HW, um_per_px=UM_PER_PX, cell_um=DENSITY_CELL_UM, frame_label=frame_label)
save(fig, "density_maps")
img0, _ = bio.load_frame(DATA_DIR, m["frame"].iloc[0])
fig = bst.plot_drift(m, NATIVE_HW, um_per_px=UM_PER_PX, img=img0, time_label=TIME_LABEL)
save(fig, "drift")

## 11. Shape

In [ ]:
fig = bst.plot_shape(bubbles, m, TIME_LABEL, smooth=SMOOTH)
save(fig, "shape")

## 12. Tracking check

Left: bubbles per frame as detected by the model, and after tracking with the missed ones bridged. The tracked count
should be smoother. Right: how many bubbles per frame were rescued or interpolated. A frame with many
interpolated bubbles is one the model handled badly. Look at it in napari.

In [ ]:
if USE_TRACKING:
    fig = bt.plot_tracking_qc(tframes, TIME_LABEL, smooth=SMOOTH)
    save(fig, "tracking_check")

## 13. Individual bubbles: lifetimes, paths and events

Lifetimes: how long each bubble is followed. Bubbles that exist at the start or end of the train are "cut", so their
real lifetime is longer. Paths: the trajectory of every bubble followed for at least `MIN_FRAMES_PATH` frames.
Event map: where bubbles merged, dissolved or appeared.

In [ ]:
if USE_TRACKING:
    fig, ax = plt.subplots(figsize=(8, 4.2))
    bt.plot_lifetimes(tracks, FRAME_INTERVAL_US, ax=ax)
    save(fig, "lifetimes")
    fig = bt.plot_trajectories(bubbles, tracks, img0, events=events, um_per_px=UM_PER_PX, min_frames=MIN_FRAMES_PATH)
    save(fig, "trajectories_events")

## 14. Mergers, dissolution and new bubbles over time

Event rates per µs. Merged: a bubble ended inside an overlapping one that grew by its volume (coalescence; each
merger removes one bubble). Dissolved: it shrank, then vanished. Appeared / emerged: new bubbles, either nucleated or
visible for the first time. Right: the volume balance of the mergers. Values near 1 mean gas volume (∝ r³) is conserved,
and the shaded band is what counts as a merger.

In [ ]:
if USE_TRACKING:
    fig = bt.plot_event_rates(events, tframes, TIME_LABEL, FRAME_INTERVAL_US, bin_frames=EVENT_BIN_FRAMES,
                              tcfg=bt.TrackConfig(**TRACK))
    save(fig, "events")
    display(events.groupby(["kind", "event"]).size().rename("count").to_frame())

## 15. Growth of individual bubbles

Left: growth rate (slope of radius over time) of each bubble against its size. If small bubbles shrink while large ones
grow, gas diffuses from small to large bubbles (Ostwald ripening). Right: the radius history of the bubbles followed
longest. A sudden jump with an orange dot is another bubble merging into it (coalescence).

In [ ]:
if USE_TRACKING:
    fig = bt.plot_growth(tracks, bubbles, FRAME_INTERVAL_US, TIME_LABEL, min_frames=MIN_FRAMES_PATH)
    save(fig, "growth")

## 16. GIF of the tracked bubbles

Every bubble keeps its colour for as long as it is followed. Interpolated outlines are dashed. Colour changes on a
bubble mean the tracking lost it.

In [ ]:
if USE_TRACKING and MAKE_TRACK_GIF:
    t0 = time.time()
    path, n = bt.make_track_gif(DATA_DIR, bubbles, m, os.path.join(FIG_DIR, "tracks.gif"), every=GIF_EVERY,
                                fps=GIF_FPS, dpi=GIF_DPI, time_fmt=lambda r: f"train {TRAIN}  ·  {frame_label(r)}")
    print(f"GIF with {n} frames: {path}  ({os.path.getsize(path) / 1e6:.1f} MB, {time.time() - t0:.0f}s)")

## Notes

* Frames are 886 ns apart, so changes within one train are small. The running mean separates real trends from
  frame-to-frame scatter of the model. The standard deviation in section 4 gives the size of that scatter.
* Tracking relies on bubbles changing gradually between frames. It was tested on synthetic trains made from your
  reviewed outlines, with deformation, motion, mergers, dissolution and missed detections. There it kept one identity
  for ~99% of bubbles (~95% with 5× faster motion and strong warping), bridged ~95% of the missed frames, and found
  almost all mergers (with the right partner) and dissolutions.
* Bubbles cut by the image edge are measured from their visible arc (as before). When only a small piece is visible
  (bubble leaving the view), that size estimate jumps from frame to frame, and so does the bubble's radius history.
* A merger of a very small bubble into a large one changes the large one's volume too little to check. These are
  counted as mergers when the small bubble ends inside the large one. `events.csv` has the volume ratio of every
  merger.
* Unreviewed frames are raw model output. Spot-check a few in napari, especially if a plot shows sudden jumps.
* To compare trains, run this notebook with another `TRAIN`. Each train gets its own folder.